# 05 — Fusion module (Gated Stream Attention Fusion, GSAF)

Defines the **EVE-RAG fusion architecture** that combines the three complementary
streams into a single OOC predictor.

## Why GSAF (the novelty claim)

The three streams in EVE-RAG produce *heterogeneous* signals whose reliability
varies per sample:

- **Stream A** (entity grounding) is informative when local images exist
  (VERITE / MMFakeBench) but provides no visual grounding for NewsCLIPpings.
- **Stream B** (CLIP representation) is the strongest in-distribution but its
  AUROC degrades on OOD (VERITE).
- **Stream C** (NLI evidence stance) is strong on MMFakeBench after threshold
  tuning but weak on VERITE because of its 32% evidence coverage.

A fixed-weight fusion (concat + MLP) treats all streams as equally trustworthy
for every sample. **GSAF instead learns a per-sample gate** over the three
streams, conditioned on the streams' own representations. This lets the model
down-weight unreliable streams sample-by-sample — which is exactly the
*similarity-vs-factuality* problem identified in the recent OOC literature.

## Architecture

```
  Stream A features  ─── Linear+LN+ReLU+Drop ──┐
  Stream B features  ─── Linear+LN+ReLU+Drop ──┤── concat ── Gate(MLP→softmax(3))
  Stream C features  ─── Linear+LN+ReLU+Drop ──┘                     │
                                ↓                                    │
                         (3 stream embeds, each h-dim) ──── weighted sum ──── Classifier ──── logit
```

## What this notebook does

1. Loads Stream A / B / C feature tables and merges on `pair_id`.
2. Identifies which columns belong to which stream (the *feature schema*).
3. Drops constant / all-NaN columns, imputes missing values (zero-impute after
   per-train-split standardization is applied later in training).
4. Saves the merged feature matrix to `cache/fusion/fusion_features_all.csv`
   and the schema to `cache/fusion/feature_schema.json`.
5. Defines the GSAF model classes and writes them to `src/models/fusion.py`
   so the training notebook can import them.
6. Smoke-tests the model with a small forward pass.

The training notebook (`fusion_training.ipynb`) consumes the artefacts produced
here.


In [1]:
# === ENVIRONMENT ===
from pathlib import Path
import sys, os, json, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"Python      : {sys.version.split()[0]}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists() and (q / "src").exists():
            return q
    raise RuntimeError("Could not find repo root.")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from src.utils.paths import (
    STREAM_A_CACHE, STREAM_B_CACHE, STREAM_C_CACHE, FUSION_CACHE,
    ALL_MANIFEST,
)

FUSION_CACHE.mkdir(parents=True, exist_ok=True)
print("\nROOT          :", ROOT)
print("FUSION_CACHE  :", FUSION_CACHE)


Python      : 3.13.13
NumPy       : 2.4.4
Pandas      : 3.0.2
PyTorch     : 2.6.0+cu124
CUDA avail  : True
GPU         : NVIDIA GeForce RTX 4060 Ti
VRAM        : 8.6 GB

ROOT          : E:\eve-rag
FUSION_CACHE  : E:\eve-rag\cache\fusion


In [2]:
# === CONFIG ===
# Candidate paths for each stream's feature table. The first existing one wins.
STREAM_A_CANDIDATES = [
    STREAM_A_CACHE / "entity_scorer" / "features_all.csv",
    STREAM_A_CACHE / "results" / "stream_a_features_all.csv",
]
STREAM_B_CANDIDATES = [
    STREAM_B_CACHE / "final" / "stream_b_final_features.csv",
    STREAM_B_CACHE / "lora_stage2" / "stage2_features_all.csv",
    STREAM_B_CACHE / "features" / "clip_features_all.csv",
]
STREAM_C_CANDIDATES = [
    STREAM_C_CACHE / "stance_scores" / "stream_c_stance_scores_all.csv",
    STREAM_C_CACHE / "results" / "stream_c_features_all.csv",
]

# Columns that are NEVER features (ids, labels, metadata).
NON_FEATURE_COLS = {
    "pair_id", "image_id", "label", "label_name", "raw_label",
    "dataset", "split",
    "image_path", "raw_image_path", "has_image",
    "caption", "title", "topic", "source", "timestamp",
    "source_image_id", "source_text_id", "original_id",
    "clip_image_key", "clip_text_key",
    "text_source", "image_source", "fake_cls", "notes",
    "evidence_text", "premise", "hypothesis",   # Stream C raw text
    "entities", "entity_types",                 # Stream A raw lists
}

HIDDEN_DIM = 32   # per-stream embedding dim
SEED = 42


In [3]:
# === LOAD STREAM FEATURE TABLES ===
def first_existing(cands, label):
    for p in cands:
        if Path(p).exists():
            print(f"  ✓ {label}: {p}")
            return p
    raise FileNotFoundError(
        f"None of the candidate {label} paths exist: {[str(c) for c in cands]}"
    )

p_a = first_existing(STREAM_A_CANDIDATES, "Stream A")
p_b = first_existing(STREAM_B_CANDIDATES, "Stream B")
p_c = first_existing(STREAM_C_CANDIDATES, "Stream C")

df_a = pd.read_csv(p_a, low_memory=False)
df_b = pd.read_csv(p_b, low_memory=False)
df_c = pd.read_csv(p_c, low_memory=False)
df_a["pair_id"] = df_a["pair_id"].astype(str)
df_b["pair_id"] = df_b["pair_id"].astype(str)
df_c["pair_id"] = df_c["pair_id"].astype(str)

print(f"\n  Stream A: {df_a.shape}")
print(f"  Stream B: {df_b.shape}")
print(f"  Stream C: {df_c.shape}")

# Manifest carries the labels and dataset/split tags.
mf = pd.read_csv(ALL_MANIFEST, low_memory=False, usecols=[
    "pair_id", "dataset", "split", "label", "has_image"
])
mf["pair_id"] = mf["pair_id"].astype(str)
print(f"  Manifest: {mf.shape}")


  ✓ Stream A: E:\eve-rag\cache\stream_a\entity_scorer\features_all.csv
  ✓ Stream B: E:\eve-rag\cache\stream_b\final\stream_b_final_features.csv
  ✓ Stream C: E:\eve-rag\cache\stream_c\stance_scores\stream_c_stance_scores_all.csv

  Stream A: (97361, 17)
  Stream B: (97361, 15)
  Stream C: (97361, 13)
  Manifest: (97361, 5)


In [4]:
# === IDENTIFY FEATURE COLUMNS PER STREAM ===
def numeric_feature_cols(df, prefix):
    """Return numeric columns that are not in NON_FEATURE_COLS."""
    cands = [c for c in df.columns if c not in NON_FEATURE_COLS]
    keep = []
    for c in cands:
        if pd.api.types.is_numeric_dtype(df[c]):
            keep.append(c)
        else:
            # Try to coerce; if it's mostly numeric, keep it.
            coerced = pd.to_numeric(df[c], errors="coerce")
            if coerced.notna().mean() >= 0.95:
                df[c] = coerced
                keep.append(c)
    return keep

cols_a = numeric_feature_cols(df_a, "a")
cols_b = numeric_feature_cols(df_b, "b")
cols_c = numeric_feature_cols(df_c, "c")

print(f"Stream A features ({len(cols_a)}): {cols_a[:8]}{'...' if len(cols_a) > 8 else ''}")
print(f"Stream B features ({len(cols_b)}): {cols_b[:8]}{'...' if len(cols_b) > 8 else ''}")
print(f"Stream C features ({len(cols_c)}): {cols_c[:8]}{'...' if len(cols_c) > 8 else ''}")

if len(cols_a) == 0 or len(cols_b) == 0 or len(cols_c) == 0:
    raise RuntimeError(
        "One of the streams has zero numeric feature columns after filtering. "
        "Inspect the loaded tables and adjust NON_FEATURE_COLS if needed."
    )


Stream A features (12): ['n_entities', 'n_person', 'n_gpe', 'n_org', 'max_grounding', 'min_grounding', 'mean_grounding', 'person_max']...
Stream B features (11): ['clip_cosine', 'l2_dist', 'absdiff_mean', 'absdiff_max', 'prod_mean', 'text_norm', 'image_norm', 'has_real_image_emb']...
Stream C features (7): ['support_score', 'contradiction_score', 'neutral_score', 'stance_similarity', 'caption_len', 'evidence_len', 'len_ratio']


In [5]:
# === MERGE STREAMS INTO ONE FEATURE TABLE ===
# Disambiguate column names so a Stream B column called 'score' doesn't clash
# with a Stream A column called 'score'. Prefix every feature with its stream tag.
def prefix_cols(df, cols, prefix):
    rename = {c: f"{prefix}__{c}" for c in cols}
    sub = df[["pair_id", *cols]].rename(columns=rename)
    new_cols = [rename[c] for c in cols]
    return sub, new_cols

a_sub, a_feats = prefix_cols(df_a, cols_a, "a")
b_sub, b_feats = prefix_cols(df_b, cols_b, "b")
c_sub, c_feats = prefix_cols(df_c, cols_c, "c")

merged = (mf
          .merge(a_sub, on="pair_id", how="left")
          .merge(b_sub, on="pair_id", how="left")
          .merge(c_sub, on="pair_id", how="left"))

print(f"Merged feature table: {merged.shape}")
print(f"  Stream A coverage : {merged[a_feats[0]].notna().mean()*100:.2f}%")
print(f"  Stream B coverage : {merged[b_feats[0]].notna().mean()*100:.2f}%")
print(f"  Stream C coverage : {merged[c_feats[0]].notna().mean()*100:.2f}%")

# Drop columns that are entirely NaN (would be useless) and constant columns.
def usable(col):
    s = merged[col]
    if s.notna().sum() == 0:
        return False
    if s.nunique(dropna=True) < 2:
        return False
    return True

a_feats = [c for c in a_feats if usable(c)]
b_feats = [c for c in b_feats if usable(c)]
c_feats = [c for c in c_feats if usable(c)]
print(f"\nAfter dropping all-NaN / constant cols:")
print(f"  Stream A: {len(a_feats)} features")
print(f"  Stream B: {len(b_feats)} features")
print(f"  Stream C: {len(c_feats)} features")
print(f"  TOTAL   : {len(a_feats) + len(b_feats) + len(c_feats)} features")


Merged feature table: (97361, 35)
  Stream A coverage : 100.00%
  Stream B coverage : 100.00%
  Stream C coverage : 100.00%

After dropping all-NaN / constant cols:
  Stream A: 12 features
  Stream B: 11 features
  Stream C: 7 features
  TOTAL   : 30 features


In [6]:
# === SAVE MERGED FEATURES + SCHEMA ===
keep_cols = ["pair_id", "dataset", "split", "label", "has_image",
             *a_feats, *b_feats, *c_feats]
out_df = merged[keep_cols].copy()
out_csv = FUSION_CACHE / "fusion_features_all.csv"
out_df.to_csv(out_csv, index=False)
print(f"Wrote {out_csv}  ({out_df.shape[0]:,} rows × {out_df.shape[1]} cols)")

schema = {
    "stream_a_features": a_feats,
    "stream_b_features": b_feats,
    "stream_c_features": c_feats,
    "hidden_dim": HIDDEN_DIM,
    "n_stream_a": len(a_feats),
    "n_stream_b": len(b_feats),
    "n_stream_c": len(c_feats),
    "non_feature_cols": sorted(NON_FEATURE_COLS),
    "source_files": {
        "stream_a": str(p_a), "stream_b": str(p_b), "stream_c": str(p_c),
    },
}
schema_path = FUSION_CACHE / "feature_schema.json"
with open(schema_path, "w") as f:
    json.dump(schema, f, indent=2)
print(f"Wrote {schema_path}")


Wrote E:\eve-rag\cache\fusion\fusion_features_all.csv  (97,361 rows × 35 cols)
Wrote E:\eve-rag\cache\fusion\feature_schema.json


In [7]:
# === DEFINE GSAF AND WRITE TO src/models/fusion.py ===
FUSION_PY = ROOT / "src" / "models" / "fusion.py"
FUSION_PY.parent.mkdir(parents=True, exist_ok=True)

FUSION_SOURCE = '''"""EVE-RAG Gated Stream Attention Fusion (GSAF).

This module is auto-generated by `notebooks/05_fusion/fusion_module.ipynb`.
Re-running that notebook will overwrite this file.
"""
from __future__ import annotations
import torch
import torch.nn as nn
import torch.nn.functional as F


class StreamProjector(nn.Module):
    """Project one stream\'s raw features into a shared embedding space."""

    def __init__(self, in_dim: int, hidden_dim: int = 32, dropout: float = 0.2):
        super().__init__()
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class GatedStreamAttentionFusion(nn.Module):
    """EVE-RAG fusion model.

    For each sample, compute a per-stream embedding then learn a softmax gate
    over the active streams. The gated weighted sum feeds a small classifier
    head that produces a single OOC logit.

    Args:
        stream_a_dim, stream_b_dim, stream_c_dim: number of raw input features
            per stream. Set any of them to 0 to disable that stream.
        hidden_dim: shared embedding dimension across streams.
        ablation: one of None, \'no_a\', \'no_b\', \'no_c\', \'mlp_fusion\'.
            \'no_x\' disables stream x. \'mlp_fusion\' replaces the gate
            with a vanilla concat-MLP head.
        dropout: dropout used in projectors and classifier head.
    """

    def __init__(
        self,
        stream_a_dim: int,
        stream_b_dim: int,
        stream_c_dim: int,
        hidden_dim: int = 32,
        ablation: str | None = None,
        dropout: float = 0.3,
    ):
        super().__init__()
        if ablation not in (None, "no_a", "no_b", "no_c", "mlp_fusion"):
            raise ValueError(f"Unknown ablation: {ablation}")
        self.ablation = ablation
        self.hidden_dim = hidden_dim

        use_a = stream_a_dim > 0 and ablation != "no_a"
        use_b = stream_b_dim > 0 and ablation != "no_b"
        use_c = stream_c_dim > 0 and ablation != "no_c"
        if not (use_a or use_b or use_c):
            raise ValueError("At least one stream must be enabled.")

        self.use_a, self.use_b, self.use_c = use_a, use_b, use_c
        self.proj_a = StreamProjector(stream_a_dim, hidden_dim, dropout) if use_a else None
        self.proj_b = StreamProjector(stream_b_dim, hidden_dim, dropout) if use_b else None
        self.proj_c = StreamProjector(stream_c_dim, hidden_dim, dropout) if use_c else None
        self.n_streams = sum([use_a, use_b, use_c])

        if ablation == "mlp_fusion":
            # Ablation: concat all stream embeds and feed an MLP (no gating).
            self.gate = None
            self.classifier = nn.Sequential(
                nn.Linear(hidden_dim * self.n_streams, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 16),
                nn.ReLU(),
                nn.Linear(16, 1),
            )
        else:
            # Default GSAF: gate over n_streams.
            self.gate = nn.Sequential(
                nn.Linear(hidden_dim * self.n_streams, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, self.n_streams),
            )
            self.classifier = nn.Sequential(
                nn.Linear(hidden_dim, 16),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(16, 1),
            )

    def forward(
        self,
        x_a: torch.Tensor | None = None,
        x_b: torch.Tensor | None = None,
        x_c: torch.Tensor | None = None,
        return_gates: bool = False,
    ):
        reps = []
        if self.use_a:
            reps.append(self.proj_a(x_a))
        if self.use_b:
            reps.append(self.proj_b(x_b))
        if self.use_c:
            reps.append(self.proj_c(x_c))

        if self.ablation == "mlp_fusion":
            fused = torch.cat(reps, dim=-1)
            logits = self.classifier(fused).squeeze(-1)
            if return_gates:
                # No gates in this mode; return uniform weights for symmetry.
                B = fused.shape[0]
                gates = torch.full((B, self.n_streams), 1.0 / self.n_streams,
                                   device=fused.device)
                return logits, gates
            return logits

        cat = torch.cat(reps, dim=-1)               # [B, h*n]
        gate_logits = self.gate(cat)                # [B, n]
        gates = F.softmax(gate_logits, dim=-1)      # [B, n]
        stacked = torch.stack(reps, dim=1)          # [B, n, h]
        fused = (gates.unsqueeze(-1) * stacked).sum(dim=1)  # [B, h]
        logits = self.classifier(fused).squeeze(-1)
        if return_gates:
            return logits, gates
        return logits


def build_fusion_model(
    n_a: int,
    n_b: int,
    n_c: int,
    hidden_dim: int = 32,
    ablation: str | None = None,
    dropout: float = 0.3,
) -> GatedStreamAttentionFusion:
    """Convenience constructor used by training / eval notebooks."""
    return GatedStreamAttentionFusion(
        stream_a_dim=n_a, stream_b_dim=n_b, stream_c_dim=n_c,
        hidden_dim=hidden_dim, ablation=ablation, dropout=dropout,
    )
'''

with open(FUSION_PY, "w", encoding="utf-8") as f:
    f.write(FUSION_SOURCE)
print(f"Wrote {FUSION_PY}")


Wrote E:\eve-rag\src\models\fusion.py


In [8]:
# === SMOKE TEST: import and forward-pass GSAF ===
# Force a clean re-import in case the notebook is re-run.
import importlib, sys
if "src.models.fusion" in sys.modules:
    importlib.reload(sys.modules["src.models.fusion"])
from src.models.fusion import build_fusion_model, GatedStreamAttentionFusion

torch.manual_seed(SEED)
n_a, n_b, n_c = len(a_feats), len(b_feats), len(c_feats)

# Default GSAF
model = build_fusion_model(n_a, n_b, n_c, hidden_dim=HIDDEN_DIM)
n_params = sum(p.numel() for p in model.parameters())
print(f"GSAF default     : {n_params:,} params")

B = 8
x_a = torch.randn(B, n_a)
x_b = torch.randn(B, n_b)
x_c = torch.randn(B, n_c)
logits, gates = model(x_a, x_b, x_c, return_gates=True)
assert logits.shape == (B,), f"unexpected logits shape: {logits.shape}"
assert gates.shape == (B, 3), f"unexpected gates shape: {gates.shape}"
assert torch.allclose(gates.sum(dim=-1), torch.ones(B), atol=1e-5), "gates do not sum to 1"
print(f"  forward OK     : logits {tuple(logits.shape)}, gates {tuple(gates.shape)}")
print(f"  mean gate (A,B,C) on random input: "
      f"{gates.mean(dim=0).detach().numpy().round(3).tolist()}")

# Each ablation
for abl in ("no_a", "no_b", "no_c", "mlp_fusion"):
    m = build_fusion_model(n_a, n_b, n_c, hidden_dim=HIDDEN_DIM, ablation=abl)
    out = m(x_a, x_b, x_c)
    assert out.shape == (B,), f"{abl}: unexpected shape {out.shape}"
    print(f"  ablation '{abl:10s}' OK  ({sum(p.numel() for p in m.parameters()):,} params)")

print("\nFusion module ready. Run fusion_training.ipynb next.")


GSAF default     : 8,164 params
  forward OK     : logits (8,), gates (8, 3)
  mean gate (A,B,C) on random input: [0.2939999997615814, 0.38600000739097595, 0.3199999928474426]
  ablation 'no_a      ' OK  (5,571 params)
  ablation 'no_b      ' OK  (5,603 params)
  ablation 'no_c      ' OK  (5,731 params)
  ablation 'mlp_fusion' OK  (8,065 params)

Fusion module ready. Run fusion_training.ipynb next.
